# PFQ - Aula 41 - Correlação WIN e WDO.ipynb

In [17]:
# Carregando as bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tvDatafeed import TvDatafeed, Interval
import ta
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import kaleido
from PIL import Image

get_ipython().run_line_magic("matplotlib", "inline")
import warnings
warnings.filterwarnings("ignore")

In [18]:
# Carregando a base de dados e fazendo os checks iniciais

# Carrega as bases
tv = TvDatafeed()
ticker1 = "WIN1!"
df1 = tv.get_hist(symbol=ticker1, exchange="BMFBOVESPA", interval=Interval.in_daily, n_bars=10000)
 
ticker2 = "WDO1!" 
df2 = tv.get_hist(symbol=ticker2, exchange="BMFBOVESPA", interval=Interval.in_daily, n_bars=10000)

# Parâmetros do algoritmo
periodos = 1
p_mm = 10


# df1
df1["Retorno"] = df1["close"].pct_change(periodos)
df1["Fech_Ant"] = df1["close"].shift(1)

# df2
df2["Retorno"] = df2["close"].pct_change(periodos)
df2["Fech_Ant"] = df2["close"].shift(1)

In [19]:
#### df1

# Cria uma média móvel e sua distância
df1["mm"] = df1["close"].rolling(p_mm).mean()
df1["dist_mm"] = round((df1["close"]/df1["mm"]-1), 3)

# Inicializa os indicadores
indicator_rsi14 = ta.momentum.RSIIndicator(close = df1["close"], window = 14)
indicator_rsi2 = ta.momentum.RSIIndicator(close = df1["close"], window = 2)
indicator_atr14 = ta.volatility.AverageTrueRange(high = df1["high"], low = df1["low"]
                                                  , close = df1["close"], window = 14)
# Cria as variáveis
df1["RSI14"] = indicator_rsi14.rsi()
df1["RSI2"] = indicator_rsi2.rsi()
df1["ATR14"] = indicator_atr14.average_true_range()
df1 = df1.dropna(axis = 0) 
df1["Return_Acc"] = df1["Retorno"].cumsum(axis = 0)

#### df2

# Cria uma média móvel e sua distância
df2["mm"] = df2["close"].rolling(p_mm).mean()
df2["dist_mm"] = round((df2["close"]/df2["mm"]-1), 3)

# Inicializa os indicadores
indicator_rsi14 = ta.momentum.RSIIndicator(close = df2["close"], window = 14)
indicator_rsi2 = ta.momentum.RSIIndicator(close = df2["close"], window = 2)
indicator_atr14 = ta.volatility.AverageTrueRange(high = df2["high"], low = df2["low"]
                                                  , close = df2["close"], window = 14)

# Cria as variáveis
df2["RSI14"] = indicator_rsi14.rsi()
df2["RSI2"] = indicator_rsi2.rsi()
df2["ATR14"] = indicator_atr14.average_true_range()
df2 = df2.dropna(axis = 0) 
df2["Return_Acc"] = df2["Retorno"].cumsum(axis = 0)

In [20]:
data_frame = pd.merge(df1[["close","RSI14"]]
               , df2[["close","RSI14"]]
               , left_index = True
               , right_index = True)

data_frame.columns = [ticker1, "RSI14_" + ticker1, ticker2, "RSI14_" + ticker2]

In [21]:
data_frame

,WIN1!,RSI14_WIN1!,WDO1!,RSI14_WDO1!
datetime,,,,
2011-10-27 13:00:00,59910.0,66.683612,1749.963,28.647440
2011-10-28 13:00:00,60136.0,67.230742,1723.495,24.325756
2011-10-31 12:00:00,58925.0,61.411060,1736.618,29.966900
2011-11-01 12:00:00,57938.0,57.074631,1784.994,45.959528
2011-11-03 12:00:00,58647.0,59.298018,1780.646,44.965630
...,...,...,...,...
2026-05-22 14:00:00,177777.0,35.407466,5026.718,50.348954
2026-05-25 14:00:00,179259.0,39.151536,5021.610,49.643117
2026-05-26 14:00:00,177775.0,36.848245,5035.457,51.623012


In [ ]:
data_frame["Corr_50"] = data_frame.iloc[:, 0].rolling(50).corr(data_frame.iloc[:, 2])
print(np.mean(data_frame["Corr_50"]))

data_frame["Corr_100"] = data_frame.iloc[:, 0].rolling(100).corr(data_frame.iloc[:, 2])
print(np.mean(data_frame["Corr_100"]))

data_frame["Corr_200"] = data_frame.iloc[:, 0].rolling(200).corr(data_frame.iloc[:, 2])
print(np.mean(data_frame["Corr_200"]))

data_frame = data_frame.dropna(axis = 0) 

-0.5645600060935746
-0.6213145161155041
-0.4860579724361443


In [30]:
data_frame

,WIN1!,RSI14_WIN1!,WDO1!,RSI14_WDO1!,Corr_50,Corr_100,Corr_200
datetime,,,,,,,
2014-03-28 13:00:00,49998.0,66.878572,2264.804,27.768517,-0.568056,-0.382375,-0.351343
2014-03-31 14:00:00,50509.0,69.191945,2284.984,35.304558,-0.545055,-0.406641,-0.357054
2014-04-01 14:00:00,50450.0,68.596211,2281.774,34.684657,-0.519407,-0.449392,-0.363975
2014-04-02 14:00:00,51917.0,74.479816,2287.560,36.837485,-0.506793,-0.486943,-0.366742
2014-04-03 14:00:00,51568.0,71.068525,2298.707,40.880360,-0.485378,-0.510451,-0.373877
...,...,...,...,...,...,...,...
2026-05-22 14:00:00,177777.0,35.407466,5026.718,50.348954,-0.736679,-0.482173,-0.792771
2026-05-25 14:00:00,179259.0,39.151536,5021.610,49.643117,-0.715237,-0.504299,-0.789112
2026-05-26 14:00:00,177775.0,36.848245,5035.457,51.623012,-0.691322,-0.532725,-0.785392


In [31]:
fig = make_subplots(rows = 4, cols = 1,
                    shared_xaxes = True,
                    vertical_spacing = 0.08)

fig.add_trace(go.Scatter(x = data_frame.index, y = round(data_frame.iloc[:, 0], 3)
                         , name = ticker1, line = dict(color = "black"))
              , row = 1, col = 1)

fig.add_trace(go.Scatter(x = data_frame.index, y = round(data_frame.iloc[:, 2], 3)
                         , name = ticker2, line = dict(color = "red"))
              , row = 2, col = 1)

fig.add_trace(go.Scatter(x = data_frame.index, y = round(data_frame.iloc[:, 4], 3)
                         , name = "Corr (50)", line = dict(color = "orange"))
              , row = 3, col = 1)

fig.add_trace(go.Scatter(x = data_frame.index, y = round(data_frame.iloc[:, 5], 3)
                         , name = "Corr (100)", line = dict(color = "blue"))
              , row = 3, col = 1)

fig.add_trace(go.Scatter(x = data_frame.index, y = round(data_frame.iloc[:, 6], 3)
                         , name = "Corr (200)", line = dict(color = "black"))
              , row = 3, col = 1)

fig.add_trace(go.Scatter(x = data_frame.index, y = round(data_frame.iloc[:, 1], 3)
                         , name = "RSI 14 " + ticker1, line = dict(color = "black"))
              , row = 4, col = 1)

fig.add_trace(go.Scatter(x = data_frame.index, y = round(data_frame.iloc[:, 3], 3)
                         , name = "RSI 14 " + ticker2, line = dict(color = "red"))
              , row = 4, col = 1)

fig.update_layout(height = 1200, width = 900
                  , title_text = "Estudo de correlação - outspokenmarket.com/omnp"
                  , font_color = "blue"
                  , title_font_color = "black"
                  , xaxis4_title = "Year"
                  , yaxis_title = ticker1
                  , yaxis2_title = ticker2
                  , yaxis3_title = "Moving Correlation"
                  , yaxis4_title = "RSI 14"
                  , legend_title = "Study objects"
                  , font = dict(size = 15, color = "Black")
                 )

fig.update_layout(
    xaxis = dict(
        rangeselector = dict(
            buttons = [
                dict(count = 6,
                     label = "6m",
                     step = "month",
                     stepmode = "backward"),
                dict(count = 1,
                     label = "1y",
                     step = "year",
                     stepmode = "backward"),
                dict(step = "all")
            ]),
        type = "date")
    , xaxis4_rangeslider_visible = True
    , xaxis4_type = "date"
    , yaxis = dict(autorange = True, fixedrange = False)
    , yaxis2 = dict(autorange = True, fixedrange = False)
    , yaxis3 = dict(autorange = True, fixedrange = False)
    , yaxis4 = dict(autorange = True, fixedrange = False)
    )

#file1 = "vol_charts/" + "study - " + ticker1 + "_" + ticker2 + ".png"
#fig.write_image(file1)

fig.show()

In [ ]:
labels = ["Corr 50", "Corr 100", "Corr 200"]
corrs = [round(data_frame["Corr_50"][-1], 2)
        , round(data_frame["Corr_100"][-1], 2)
        , round(data_frame["Corr_200"][-1], 2)]
colors = ["orange", "blue", "black"]

fig = go.Figure([go.Bar(x = labels, y = corrs
                     , text = corrs
                     , textposition = "auto"
                     , marker_color = colors)])

fig.update_layout(height = 600, width = 800
                  , title_text = "Last Close Moving Correlation " + ticker1 + " x " + ticker2
                  , font_color = "blue"
                  , title_font_color = "black"
                  , xaxis_title = "Ativos"
                  , yaxis_title = "Moving Correlation"
                  , legend_title = "Tickers"
                  , font = dict(size = 15, color = "Black")
                 )
#file2 = "vol_charts/" + "corr - " + ticker1 + "_" + ticker2 + "-" + data_frame.index[-1].strftime('%Y-%m-%d') +".png"
#fig.write_image(file2)
fig.show()

In [25]:
labels = ["RSI 14 " + ticker1, "RSI 14 " + ticker2]
corrs = [round(data_frame.iloc[:, 1][-1], 2)
        , round(data_frame.iloc[:, 3][-1], 2)
        ]
colors = ["orange", "blue"]

fig = go.Figure([go.Bar(x = labels, y = corrs
                     , text = corrs
                     , textposition = "auto"
                     , marker_color = colors)])

fig.update_layout(height = 600, width = 800
                  , title_text = "Last RSI 14 " + ticker1 + " x " + ticker2
                  , font_color = "blue"
                  , title_font_color = "black"
                  , xaxis_title = "Ativos"
                  , yaxis_title = "RSI 14"
                  , legend_title = "Tickers"
                  , font = dict(size = 15, color = "Black")
                 )
#file3 = "vol_charts/" + "rsi - " + ticker1 + "_" + ticker2 + "-" + data_frame.index[-1].strftime('%Y-%m-%d') +".png"
#fig.write_image(file3)
fig.show()

In [ ]:
# Se quiser um report em pdf

'''
i1 = Image.open(file1).convert("RGB")
i2 = Image.open(file2).convert("RGB")
i3 = Image.open(file3).convert("RGB")

i_lista = [i2, i3] #o i1 não vai aqui para não duplicar no relatório

i1.save("vol_charts/" + ticker1 + ticker2 + "report.pdf"
        , save_all = True, append_images = i_lista)

'''